In [ ]:
# Run from the repo root so `src` and `data/` resolve (this notebook lives in notebooks/).
# Exploratory only: the production labels come from scripts/run_tagging.py and
# scripts/apply_names_policy.py; the "Full run" cell below predates AnE-LID and is historical.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from datasets import load_dataset

# Run this to save dataset locally to cache
def save_to_cache():
    # Login using e.g. `huggingface-cli login` to access this dataset
    ds = load_dataset("Shelton1013/SwitchLingua_text")
    return ds

In [ ]:
import pandas as pd

ds = save_to_cache()

In [ ]:
# ---------------------------------------------------------------------------
# EDA: value frequencies for the 13 columns of interest
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

# pd.DataFrame(ds) on a DatasetDict gives a 1-column frame named 'train'.
# Unwrap the split to get the actual table.
df = ds["train"].to_pandas()
print(f"df.shape = {df.shape}\n")

CAT_COLS = [
    "topic", "tense", "perspective", "cs_ratio", "gender", "age",
    "education_level", "first_language", "second_language",
    "conversation_type", "cs_function", "cs_type",
]
NUM_COL = "score"

# Columns with a meaningful natural order - keep it instead of sorting by count.
ORDINAL_ORDERS = {
    "tense": ["Past", "Present", "Future"],
    "cs_ratio": ["30%", "50%", "70%"],
    "age": ["8-17", "18-25", "26-35", "56-65", "66+"],
    "education_level": ["High School", "College", "Master", "Doctor"],
}

SERIES = "#2a78d6"   # one hue: every subplot is a single series (frequency)
INK_MUTED = "#52514e"


def freq_table(s: pd.Series, order=None) -> pd.DataFrame:
    """Counts + percentages, in natural order if given else by frequency."""
    counts = s.value_counts(dropna=False)
    if order is not None:
        counts = counts.reindex(order).fillna(0).astype(int)
    out = counts.to_frame("count")
    out["pct"] = (out["count"] / len(s) * 100).round(2)
    return out


# --- 1. Frequency tables ----------------------------------------------------
for col in CAT_COLS:
    s = df[col]
    print(f"=== {col} | dtype={s.dtype} | nunique={s.nunique()} | nulls={s.isna().sum()}")
    print(freq_table(s, ORDINAL_ORDERS.get(col)).to_string(), "\n")

# score is continuous (229 distinct floats), so summarise and bin it instead.
s = df[NUM_COL]
print(f"=== {NUM_COL} | dtype={s.dtype} | nunique={s.nunique()} | nulls={s.isna().sum()}")
print(s.describe().to_string(), "\n")
score_bins = pd.cut(s, bins=10).value_counts().sort_index().to_frame("count")
score_bins["pct"] = (score_bins["count"] / len(s) * 100).round(2)
print(score_bins.to_string(), "\n")


# --- 2. Charts --------------------------------------------------------------
fig, axes = plt.subplots(5, 3, figsize=(16, 22))
axes = axes.ravel()

for ax, col in zip(axes, CAT_COLS):
    tbl = freq_table(df[col], ORDINAL_ORDERS.get(col))
    if col not in ORDINAL_ORDERS:
        tbl = tbl.sort_values("count", ascending=False)

    ypos = list(range(len(tbl)))
    ax.barh(ypos, tbl["count"], color=SERIES, height=0.62, zorder=3)
    ax.set_yticks(ypos)
    ax.set_yticklabels(tbl.index.astype(str), fontsize=9)
    # Descending ylim inverts the axis (largest/first on top) and keeps bar
    # thickness consistent across subplots of different cardinality.
    ax.set_ylim(max(len(tbl), 5) - 0.4, -0.6)
    ax.set_xlim(0, tbl["count"].max() * 1.32)

    for y, (n, pct) in enumerate(zip(tbl["count"], tbl["pct"])):
        ax.text(n + tbl["count"].max() * 0.02, y, f"{n:,} ({pct:.1f}%)",
                va="center", fontsize=8, color=INK_MUTED)

    ax.set_title(f"{col}  (n={tbl.shape[0]})", fontsize=11, loc="left", pad=8)
    ax.xaxis.set_visible(False)          # bar ends are directly labelled
    for side in ("top", "right", "bottom"):
        ax.spines[side].set_visible(False)
    # Stop the axis line at the last bar rather than trailing into the padding.
    ax.spines["left"].set_bounds(-0.6, len(tbl) - 0.4)
    ax.tick_params(left=False)

# score keeps its x-axis - the continuous scale is the point.
ax = axes[len(CAT_COLS)]
ax.hist(s, bins=50, color=SERIES, zorder=3)
ax.set_title(f"{NUM_COL}  (continuous, {s.nunique()} distinct)",
             fontsize=11, loc="left", pad=8)
ax.set_xlabel("score", fontsize=9, color=INK_MUTED)
ax.grid(axis="y", alpha=0.3, zorder=0)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

for ax in axes[len(CAT_COLS) + 1:]:
    ax.set_visible(False)

fig.tight_layout(pad=2.0)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Run this FIRST if a tagging cell is taking more than a minute.
# ---------------------------------------------------------------------------
from src.bert_tagger import diagnose
diagnose()


In [ ]:
# ---------------------------------------------------------------------------
# Code-switch tagging pipeline (see data_pipeline_plan.md)
# Stratified sample + validation against the dataset's own cs_ratio_result.
# ---------------------------------------------------------------------------
import sys, time
sys.path.insert(0, ".")

import pandas as pd

from src.routing import route, tagger_quality
from src.segments import tag_row
from src.validate import parse_computed_ratio

N_PER_LANG = 150

# groupby().sample() keeps the grouping column; groupby().apply() drops it in pandas 3.
sample = (df.groupby("first_language", group_keys=False)
            .sample(n=N_PER_LANG, random_state=0)
            .reset_index(drop=True))
print(f"sample: {sample.shape[0]} rows across {sample.first_language.nunique()} languages")

t0 = time.time()
tagged = pd.DataFrame([
    tag_row(r.data_generation_result, r.first_language, r.second_language)
    for r in sample.itertuples()
])
elapsed = time.time() - t0
print(f"tagged in {elapsed:.1f}s ({elapsed / len(sample) * 1000:.1f} ms/row) "
      f"-> full {len(df):,} rows ~ {elapsed / len(sample) * len(df) / 60:.1f} min")

tagged_sample = sample.join(tagged)
tagged_sample["reported_l2"] = [
    parse_computed_ratio(c, l)
    for c, l in zip(tagged_sample.cs_ratio_result, tagged_sample.second_language)
]
tagged_sample["quality"] = tagged_sample.first_language.map(tagger_quality)

# --- validation -------------------------------------------------------------
# The six script-disjoint languages are tagged exactly, so their disagreement
# with cs_ratio_result measures noise in that reference, not tagger error.
# That band is the yardstick for judging the model-routed pairs.
report = []
for lang, g in tagged_sample.groupby("first_language"):
    v = g.dropna(subset=["reported_l2"])
    report.append(dict(
        language=lang, method=g.method.iloc[0], quality=g.quality.iloc[0], n=len(g),
        mine=round(g.l2_token_ratio.mean(), 3),
        reported=round(v.reported_l2.mean(), 3),
        diff=round(g.l2_token_ratio.mean() - v.reported_l2.mean(), 3),
        corr=round(v.l2_token_ratio.corr(v.reported_l2), 3),
        unknown=round(g.unknown_rate.mean(), 3),
        switches=round(g.n_switches.mean(), 1),
        mean_run=round(g.l2_runs.map(lambda r: sum(r) / len(r) if r else 0).mean(), 1),
        max_run=int(g.max_run.max()),
    ))
report = pd.DataFrame(report).sort_values(["method", "diff"])
print("\n=== per-language validation ===")
print(report.to_string(index=False))

exact_band = report.loc[report.method == "unicode", "diff"].abs().max()
print(f"\nexact-tagger noise band: |diff| <= {exact_band:.3f}")
print("model pairs outside it:",
      report[(report.method == "model") & (report["diff"].abs() > exact_band)]
      .language.tolist())

# --- the original question: L2 run lengths ---------------------------------
trust = tagged_sample[tagged_sample.quality != "low"]
runs = trust.explode("l2_runs").dropna(subset=["l2_runs"])
runs["l2_runs"] = runs.l2_runs.astype(int)
print(f"\n=== L2 run-length distribution ({len(runs):,} switches, "
      f"quality != low) ===")
print(runs.l2_runs.describe().to_string())
print("\nrun length -> share of switches")
print((runs.l2_runs.value_counts(normalize=True).sort_index().head(12) * 100)
      .round(1).to_string())


In [ ]:
# ---------------------------------------------------------------------------
# Visual check: one tagged row per language, colour-coded by tag.
# Green/amber/red badge = measured trust tier (exact / ok / low).
# Read the red ones first - that is where the tagger is known to be wrong.
# ---------------------------------------------------------------------------
from src.inspection import show_sample, show_row

show_sample(df, n_per_language=1, max_utterances=2)


In [ ]:
# ---------------------------------------------------------------------------
# Full run -> data/tagged.parquet   (~1.5 min for all 234k rows)
# ---------------------------------------------------------------------------
from src.persist import tag_dataframe, save_tagged, load_tagged

tagged_full = tag_dataframe(df)
path = save_tagged(tagged_full)
print(f"wrote {path}  ({len(tagged_full):,} rows)")

# verify the round-trip before trusting the artifact
check = load_tagged(path)
assert len(check) == len(tagged_full)
assert all(sum(r.segment_len) == len(r.tokens) for r in check.head(500).itertuples())
print("round-trip verified")
print(check[["first_language", "quality", "n_switches", "max_run",
             "total_l2_tokens"]].head())


In [ ]:
# ---------------------------------------------------------------------------
# Analyse the tagged corpus. Pure pandas - NO kernel requirement beyond
# pandas + pyarrow, so this runs on the Anaconda kernel just fine.
# Produce the file first with:   ./.venv/bin/python scripts/run_tagging.py
# ---------------------------------------------------------------------------
import pandas as pd

tagged = pd.read_parquet("data/tagged.parquet")
print(f"{len(tagged):,} rows | tiers: {dict(tagged.quality.value_counts())}\n")

# Trustworthy subset: drop the pairs still under the 90% accuracy bar.
trust = tagged[tagged.quality != "low"]
runs = trust[["first_language", "l2_runs"]].explode("l2_runs").dropna()
runs["l2_runs"] = runs.l2_runs.astype(int)

print(f"=== L2 run lengths ({len(runs):,} switches across {len(trust):,} rows) ===")
print(runs.l2_runs.describe().round(2).to_string())

print("\n=== share of switches by run length ===")
pct = (runs.l2_runs.value_counts(normalize=True).sort_index() * 100).round(1)
print(pct.head(10).to_string())

print("\n=== mean run length by language ===")
print(runs.groupby("first_language").l2_runs.agg(["count", "mean", "median", "max"])
         .round(2).sort_values("mean").to_string())
